# 12 - Train a Retrace(λ) DQN Model Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but trains with `RetraceObjective` ([Munos et al., 2016](https://arxiv.org/abs/1606.02647)) instead of one-step `DqnObjective`:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, an action-value head **and a behavior head**.
4. Train with `RetraceObjective` and save with `push_model_to_hub`.

Retrace(λ) is off-policy, return-based Q-learning. The TD target of each transition is the delayed one-step expected backup plus a trace of later TD errors, each scaled by the product of **truncated importance ratios** `c_s = λ · min(1, π(a_s|s_s) / μ(a_s|s_s))`. `π` is the target policy — `softmax(Q / temperature)` over the delayed Q (Q as logits), the same convention as `model.get_action(temperature=)` — and `μ` is the behavior policy that produced the data. Near-on-policy transitions keep the full λ-return; strongly off-policy actions cut the trace; the clip at `1` keeps the variance bounded, so `λ = 1` is safe (the paper's Atari setting).

The dataset stores no behavior probabilities. `μ` is **learned**: a second `DiscreteActionHead` under the `behavior` key outputs logits whose `log_softmax` is `log μ(·|s)`; it is fit by negative log-likelihood of the actions in the data (inside the same objective call), and the same distribution at each step is the `μ(·|s)` the trace uses. Because it sees the same in-context history as the Q head, it can track a behavior policy that changes along a task — such as the per-episode oracle ramp in `01_collect_dataset.ipynb`.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import RetraceObjective
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead, DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-retrace-offline"          # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-retrace-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
TD_LAMBDA = 1.0                               # Retrace λ (1.0: traces are cut only by min(1, π/μ))
TARGET_TEMPERATURE = 0.1                      # softmax temperature of the target policy π = softmax(Q / T) on the delayed Q (0 = greedy)
BEHAVIOR_WEIGHT = 1.0                         # weight of the behavior head's NLL loss that learns μ
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.01                     # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves. `use_norm` (required, saved with the model) keeps (`True`) or drops (`False`) the final RMSNorm.
- `DiscreteActionValueHead` predicts one value per discrete action. `use_norm` (required) keeps (`True`) or drops (`False`) the head's input RMSNorm.
- `DiscreteActionHead` predicts one logit per discrete action. Here it is the **behavior head**: its softmax is the learned `μ(·|s)`. It reads the same pooled features as the Q head.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so Q is read from it).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. Heads are passed as a dict with caller-chosen keys — `action_value` for Q and `behavior` for μ — and `action_head="action_value"` tells `get_action` to read the Q head, so the behavior head never picks actions at inference.


In [ ]:
backbone = Qwen3Backbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained="Qwen/Qwen3-0.6B",
)

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

q_head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
)

behavior_head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=1.0,
    use_norm=True,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={"action_value": q_head, "behavior": behavior_head},
    action_head="action_value",
    reasoner=None,
    recurrence=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions, delayed_predictions)` computes the Retrace loss and metrics.
4. `AdamW` updates weights. The backbone, encoder, and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q comes from the delayed model: `delayed_model = model.delayed_copy(heads=("action_value",))` is a frozen copy of the online model carrying only the heads the target reads — the fp32 encoder, backbone, and Q head are copied. The `behavior` head is left out: Retrace reads `μ` from the online head only, so the delayed model neither runs it nor Polyak-interpolates it. After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`RetraceObjective` takes `td_lambda` (`TD_LAMBDA`), `temperature` (`TARGET_TEMPERATURE`), and `behavior_weight` (`BEHAVIOR_WEIGHT`), all required. Along a run the target is

`G_i = r_i + γ_i · ( E_π Q(s_{i+1}, ·) + c_{i+1} · (G_{i+1} − Q(s_{i+1}, a_{i+1})) )`, with `c = λ · min(1, π/μ)`,

where every target quantity — `E_π Q`, `Q(s', a')`, and `π = softmax(Q / temperature)` itself — comes from the delayed model (`π` over the head's raw Q treated as logits, so `temperature` means the same as in `get_action`), and `μ(·|s)` is the online behavior head's softmax, detached. `td_lambda=0` is the expected one-step target; `temperature=0` is the greedy target policy, i.e. Watkins's Q(λ); a higher temperature flattens `π` and cuts fewer traces. The returned loss is `td_loss + behavior_weight · behavior_loss`, where `behavior_loss` is the behavior head's negative log-likelihood of the action taken from each step, `-log μ(a|s)`. `behavior_weight=0` drops the NLL from the loss; the head is still required and its softmax is still `μ`. Metrics: `td_loss`, `behavior_loss`, `behavior_prob_mean` (μ of the taken action — rises as the behavior head fits the data), and `retrace_ratio_mean` (mean `min(1, π/μ)`; `1` is on-policy, near `0` means the traces are cut everywhere and the target is one-step).

`episode_done` and `task_done` (each `0`/`1`/`2`) map to separate discount factors exactly as in `DqnObjective`. The gamma at `i+1` multiplies both the bootstrap and the continued trace, so a `0` gamma ends the trace and a non-zero truncation gamma carries it, discounted. The trace never crosses a run break (`sequence_id` / `grouping_field`).


In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.delayed_copy(heads=("action_value",))
polyak = Polyak(model, delayed_model)
objective = RetraceObjective(
    td_lambda=TD_LAMBDA,
    temperature=TARGET_TEMPERATURE,
    behavior_weight=BEHAVIOR_WEIGHT,
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: RetraceObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        loss, metrics = objective(
            objective_data.to(device),
            out.predictions,
            delayed_out.predictions,
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_encoder=POLYAK_TAU_ENCODER,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  td={metrics['td_loss']:.4f}  "
        f"bc={metrics['behavior_loss']:.4f}  mu={metrics['behavior_prob_mean']:.3f}  "
        f"ratio={metrics['retrace_ratio_mean']:.3f}  q={metrics['q_values_mean']:.3f}"
    )
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")